In [1]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry, ModelType
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
from schemas.similarity import SearchMethod
import os


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/bilstm_attention_cosine'} models=[<SearchMethod.SBERT: 'sbert'>, <SearchMethod.LSTM: 'lstm'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 spell_max_distance=2 spell_unigram_weight=0.8 spell_data_dir='./spelling_checker/data' spell_model_dir='./spelling_checker/models' faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [3]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "dialogue", "active")
structure_dir = os.path.join(localization_dir,  "structure", "modified")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [4]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['computer/usernames', 'scene4/scene4Frontyard', 'scene1/scene1Bedroom2', 'scene6/routeA/scene6EndingRouteA', 'transitions', 'scene1/scene1Break', 'scene2/scene2Break', 'scene6/scene6Livingroom', 'dialogManager', 'computer/captions', 'scene1/scene1Bedroom1', 'scene5/scene5Bedroom', 'scene6/routeA/scene6LunchRouteA', 'computer/socialMediaScreen', 'scene3/scene3Break', 'scene2/scene2Bedroom', 'scene6/routeA/scene6BedroomRouteA1', 'scene1/scene1Classroom', 'scene6/scene6Bedroom', 'scene5/scene5Livingroom', 'scene6/routeB/scene6PoliceStationRouteB', 'scene6/routeA/scene6PortalRouteA', 'menus/creditsScene', 'scene6/routeA/scene6BedroomRouteA2', 'scene3/scene3Bedroom', 'scene7/scene7Bedroom', 'scene6/routeB/scene6BedroomRouteB', 'scene4/scene4Bedroom', 'names', 'scene4/scene4Garage', 'scene1/scene1Lunch2', 'scene4/scene4Backyard', 'scene1/scene1Lunch1', 'deviceInfo', 'computer/loginScreen', 'scene6/routeB/scene6LunchRouteB', 'scene6/routeB/scene6EndingRouteB', 'generalDialogs', 'menus/titleS

In [5]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [7]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer(ModelType.SBERT)
model_registry.build_spacy()
model_registry.build_word2vec()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)


2026-07-20 21:46:55.645 | DEBUG    | services.model_registry:_create_loader:59 - Registering sbert loader for 'es'.
2026-07-20 21:46:55.645 | DEBUG    | services.model_registry:_create_loader:59 - Registering spaCy loader for 'es'.
2026-07-20 21:46:55.645 | DEBUG    | services.model_registry:_create_loader:59 - Registering word2vec loader for 'es'.
2026-07-20 21:46:55.645 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-20 21:46:58.533 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-20 21:46:58.533 | DEBUG    | services.lazy_loader:model:16 - Loading word2vec for 'es'...
2026-07-20 21:47:10.424 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded word2vec for 'es'
2026-07-20 21:47:10.424 | DEBUG    | services.lazy_loader:model:16 - Loading spaCy for 'es'...
2026-07-20 21:47:11.066 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded spaCy for 'es'


In [8]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
    model_types=[
        SearchMethod.SBERT,
        SearchMethod.WORD2VEC_IDF
	]
)

builder.run()


2026-07-20 21:47:12.003 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 33 vectors
2026-07-20 21:47:12.287 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 33 vectors
2026-07-20 21:47:12.373 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-20 21:47:12.571 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-20 21:47:12.652 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 31 vectors
2026-07-20 21:47:12.853 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 31 vectors
2026-07-20 21:47:12.978 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 48 vectors
2026-07-20 21:47:13.177 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 48 vectors
2026-07-20 21:47:13.239 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-20 21:47:13.444 | DEBUG    | controllers.retrievers.faiss:_fit:98 - Indexed 40 vectors
2026-07-20 21:47:13.524 | DEBUG    | controllers.r

Total visited nodes: 732


In [9]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", SearchMethod.SBERT)

print(test_engine.retrievers)

test_engine.load_all()

# test_engine.load_node("scene1Bedroom1_computer1_choices_similarity")

print(test_engine.retrievers)


2026-07-20 21:47:15.960 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-20 21:47:15.964 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-20 21:47:15.965 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom1_computer2_root
2026-07-20 21:47:15.968 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-20 21:47:15.969 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Bedroom2_computer_choices2_similarity
2026-07-20 21:47:15.970 | SUCCESS  | services.node_engine:load_node:88 - Loaded node successfully.
2026-07-20 21:47:15.971 | DEBUG    | services.node_engine:load_node:70 - Loading FAISS node | method=sbert | language=es | node=scene1Classroom_part2_thanks_similarity
2026-07-20 21:47:15.972 | SUCCESS

{}
{'scene1Bedroom1_computer1_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913A2960>, 'scene1Bedroom1_computer2_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913DF140>, 'scene1Bedroom2_computer_choices2_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913DC350>, 'scene1Classroom_part2_thanks_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913DCB30>, 'scene2Break_part2_choice_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913DEA50>, 'scene3Bedroom_main_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913A0620>, 'scene4Backyard_mainConversation_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC9135EC60>, 'scene4Bedroom_phone_root': <controllers.retrievers.faiss.FaissRetriever object at 0x000002EC913B3830>, 'scene4Garage_phone1_root': <controllers.retrievers.fais

In [10]:
retriever = test_engine.get_retriever("scene1Bedroom1_computer1_choices_similarity")

retriever.search("Hola", 3)


768
768


(array([10, 12, 11], dtype=int32),
 array([0.5469646 , 0.53729975, 0.48407146], dtype=float32),
 array(['Hola. En serio?', 'Holaa, pues bien', 'Hola jaja. Supongo'],
       dtype=object))